# Session 2, Module 10: Composition vs Inheritance


This module covers:
- "Has-a" vs "Is-a" relationship
- Building a Pipeline class that composes Extractor, Transformer, Loader
- Dependency injection basics
- When to use inheritance vs composition

Data Engineering Context:
Composition creates flexible, testable pipelines where components can be
swapped without changing the pipeline code. This is the foundation of
pluggable architecture.


In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Protocol, runtime_checkable

## Inheritance: "Is-A" Relationship


In [ ]:
print("=== Inheritance: Is-A Relationship ===")


# With inheritance, subclasses ARE the parent type
class Animal:
    def speak(self):
        raise NotImplementedError


class Dog(Animal):  # Dog IS-A Animal
    def speak(self):
        return "Woof!"


class Cat(Animal):  # Cat IS-A Animal
    def speak(self):
        return "Meow!"


dog = Dog()
cat = Cat()
print(f"Dog says: {dog.speak()}")
print(f"Cat says: {cat.speak()}")
print(f"Dog is Animal: {isinstance(dog, Animal)}")

Problems with deep inheritance:
- Tight coupling
- Fragile base class problem
- Diamond problem with multiple inheritance
- Changes to base class affect all subclasses

## Composition: "Has-A" Relationship


In [ ]:
print("\n=== Composition: Has-A Relationship ===")


# With composition, objects HAVE components
class Engine:
    def __init__(self, horsepower: int):
        self.horsepower = horsepower

    def start(self):
        return f"Engine ({self.horsepower}hp) started"


class Wheels:
    def __init__(self, count: int):
        self.count = count

    def roll(self):
        return f"{self.count} wheels rolling"


class Car:  # Car HAS-A Engine and Wheels
    def __init__(self, engine: Engine, wheels: Wheels):
        self.engine = engine  # Composition
        self.wheels = wheels  # Composition

    def drive(self):
        return f"{self.engine.start()}, {self.wheels.roll()}"


# Create components
engine = Engine(200)
wheels = Wheels(4)

# Compose them into a car
car = Car(engine, wheels)
print(f"Car: {car.drive()}")

Benefits:
- Loose coupling
- Easy to swap components
- Easy to test (mock components)
- More flexible

## Practical: Etl Pipeline With Composition


In [ ]:
print("\n=== ETL Pipeline with Composition ===")


# Define component interfaces using Protocol (duck typing)
@runtime_checkable
class Extractor(Protocol):
    """Interface for data extractors."""

    def extract(self) -> list[dict]:
        """Extract data from source."""
        ...


@runtime_checkable
class Transformer(Protocol):
    """Interface for data transformers."""

    def transform(self, data: list[dict]) -> list[dict]:
        """Transform data."""
        ...


@runtime_checkable
class Loader(Protocol):
    """Interface for data loaders."""

    def load(self, data: list[dict]) -> int:
        """Load data to destination. Returns count."""
        ...


# Concrete implementations
class DatabaseExtractor:
    """Extract from database."""

    def __init__(self, connection_string: str, query: str):
        self.connection_string = connection_string
        self.query = query

    def extract(self) -> list[dict]:
        print(f"  Extracting from DB: {self.query[:30]}...")
        # Simulated data
        return [
            {"id": 1, "name": "alice", "value": 100},
            {"id": 2, "name": "bob", "value": 200},
        ]


class APIExtractor:
    """Extract from API."""

    def __init__(self, base_url: str, endpoint: str):
        self.base_url = base_url
        self.endpoint = endpoint

    def extract(self) -> list[dict]:
        print(f"  Extracting from API: {self.base_url}/{self.endpoint}")
        return [
            {"id": 3, "name": "charlie", "value": 300},
        ]


class CleaningTransformer:
    """Clean and normalize data."""

    def transform(self, data: list[dict]) -> list[dict]:
        print(f"  Cleaning {len(data)} records...")
        return [
            {**record, "name": record["name"].title()}
            for record in data
        ]


class FilterTransformer:
    """Filter records by condition."""

    def __init__(self, min_value: int):
        self.min_value = min_value

    def transform(self, data: list[dict]) -> list[dict]:
        print(f"  Filtering records with value >= {self.min_value}...")
        return [r for r in data if r.get("value", 0) >= self.min_value]


class WarehouseLoader:
    """Load to data warehouse."""

    def __init__(self, table_name: str):
        self.table_name = table_name

    def load(self, data: list[dict]) -> int:
        print(f"  Loading {len(data)} records to {self.table_name}")
        return len(data)


class FileLoader:
    """Load to file."""

    def __init__(self, file_path: str):
        self.file_path = file_path

    def load(self, data: list[dict]) -> int:
        print(f"  Writing {len(data)} records to {self.file_path}")
        return len(data)

## Pipeline Class With Composition


In [ ]:
@dataclass
class Pipeline:
    """
    ETL Pipeline using composition.

    The pipeline doesn't inherit from extractors/transformers/loaders.
    Instead, it HAS them as components that can be swapped.
    """

    name: str
    extractor: Extractor
    transformers: list[Transformer] = field(default_factory=list)
    loader: Loader = None

    def run(self) -> dict:
        """Execute the pipeline."""
        print(f"\n--- Running pipeline: {self.name} ---")

        # Extract
        data = self.extractor.extract()
        print(f"  Extracted {len(data)} records")

        # Transform (chain of transformers)
        for transformer in self.transformers:
            data = transformer.transform(data)

        # Load
        if self.loader:
            count = self.loader.load(data)
            print(f"  Loaded {count} records")
        else:
            print("  No loader configured (dry run)")

        return {
            "pipeline": self.name,
            "records_processed": len(data),
            "status": "success",
        }


# Create pipeline with different components
print("=== Pipeline 1: Database to Warehouse ===")
pipeline1 = Pipeline(
    name="db_to_warehouse",
    extractor=DatabaseExtractor("postgresql://...", "SELECT * FROM users"),
    transformers=[
        CleaningTransformer(),
        FilterTransformer(min_value=150),
    ],
    loader=WarehouseLoader("dim_users"),
)
result1 = pipeline1.run()
print(f"Result: {result1}")

print("\n=== Pipeline 2: API to File ===")
pipeline2 = Pipeline(
    name="api_to_file",
    extractor=APIExtractor("https://api.example.com", "users"),
    transformers=[CleaningTransformer()],
    loader=FileLoader("/data/output/users.json"),
)
result2 = pipeline2.run()
print(f"Result: {result2}")

## Dependency Injection


In [ ]:
print("\n=== Dependency Injection ===")


class PipelineFactory:
    """
    Factory that creates pipelines with injected dependencies.

    This is a simple form of dependency injection.
    """

    def __init__(self):
        self._extractors: dict = {}
        self._loaders: dict = {}

    def register_extractor(self, name: str, extractor: Extractor) -> None:
        """Register an extractor by name."""
        self._extractors[name] = extractor

    def register_loader(self, name: str, loader: Loader) -> None:
        """Register a loader by name."""
        self._loaders[name] = loader

    def create_pipeline(
        self,
        name: str,
        extractor_name: str,
        loader_name: str,
        transformers: list[Transformer] = None,
    ) -> Pipeline:
        """Create a pipeline with registered components."""
        extractor = self._extractors.get(extractor_name)
        loader = self._loaders.get(loader_name)

        if not extractor:
            raise ValueError(f"Unknown extractor: {extractor_name}")
        if not loader:
            raise ValueError(f"Unknown loader: {loader_name}")

        return Pipeline(
            name=name,
            extractor=extractor,
            transformers=transformers or [],
            loader=loader,
        )


# Configure factory with available components
factory = PipelineFactory()

# Register extractors
factory.register_extractor("users_db", DatabaseExtractor("postgresql://...", "SELECT * FROM users"))
factory.register_extractor("users_api", APIExtractor("https://api.example.com", "users"))

# Register loaders
factory.register_loader("warehouse", WarehouseLoader("analytics.users"))
factory.register_loader("file", FileLoader("/data/users.csv"))

# Create pipeline from factory
pipeline3 = factory.create_pipeline(
    name="users_sync",
    extractor_name="users_api",
    loader_name="warehouse",
    transformers=[CleaningTransformer()],
)

print("=== Pipeline 3: Created from Factory ===")
result3 = pipeline3.run()
print(f"Result: {result3}")

## When To Use Inheritance Vs Composition


In [ ]:
print("\n=== When to Use What ===")

print("""
USE INHERITANCE when:
├─ True "is-a" relationship (Dog IS Animal)
├─ Sharing implementation (not just interface)
├─ Framework requires it (e.g., Django models)
└─ Small, stable hierarchies

USE COMPOSITION when:
├─ "Has-a" relationship (Car HAS Engine)
├─ Need flexibility to swap components
├─ Components are reusable across classes
├─ Testing requires mocking
├─ Avoiding tight coupling
└─ Following dependency injection patterns

PREFER COMPOSITION because:
├─ More flexible (change behavior at runtime)
├─ Easier to test (inject mock components)
├─ Avoids inheritance problems (diamond, fragile base)
├─ Follows "favor composition over inheritance"
└─ Leads to loosely coupled designs

Common Pattern: Use inheritance for INTERFACES (ABC),
               composition for IMPLEMENTATION.

Example:
  # Interface via inheritance
  class Extractor(ABC):
      @abstractmethod
      def extract(self): pass

  # Implementation via composition
  class Pipeline:
      def __init__(self, extractor: Extractor):
          self.extractor = extractor  # HAS-A
""")

## Practical: Testing With Composition


In [ ]:
print("=== Testing Benefit of Composition ===")


# Mock extractor for testing
class MockExtractor:
    """Mock extractor for testing."""

    def __init__(self, mock_data: list[dict]):
        self.mock_data = mock_data
        self.extract_called = False

    def extract(self) -> list[dict]:
        self.extract_called = True
        return self.mock_data


# Mock loader for testing
class MockLoader:
    """Mock loader for testing."""

    def __init__(self):
        self.loaded_data = []
        self.load_called = False

    def load(self, data: list[dict]) -> int:
        self.load_called = True
        self.loaded_data = data
        return len(data)


# Test the pipeline with mocks
print("\nTesting pipeline with mocks:")
mock_extractor = MockExtractor([{"id": 1, "name": "test", "value": 100}])
mock_loader = MockLoader()

test_pipeline = Pipeline(
    name="test_pipeline",
    extractor=mock_extractor,
    transformers=[CleaningTransformer()],
    loader=mock_loader,
)

result = test_pipeline.run()

# Verify behavior
print(f"\nTest Results:")
print(f"  Extract called: {mock_extractor.extract_called}")
print(f"  Load called: {mock_loader.load_called}")
print(f"  Loaded data: {mock_loader.loaded_data}")
print(f"  Name transformed: {mock_loader.loaded_data[0]['name'] == 'Test'}")

## Summary


In [ ]:
print("\n=== Summary ===")
print("""
Inheritance (Is-A):
  class Dog(Animal):  # Dog IS-A Animal
      pass
  - Subclass inherits parent's code
  - Tight coupling
  - Use sparingly

Composition (Has-A):
  class Car:
      def __init__(self, engine: Engine):
          self.engine = engine  # Car HAS-A Engine
  - Objects contain other objects
  - Loose coupling
  - Easy to swap components
  - Preferred approach

Dependency Injection:
  - Pass dependencies instead of creating them
  - Makes testing easy (inject mocks)
  - Enables configuration-driven behavior

Protocol (Python 3.8+):
  from typing import Protocol

  class MyProtocol(Protocol):
      def method(self): ...
  - Structural subtyping (duck typing with types)
  - No inheritance needed
  - Just implement the methods

Best Practices:
  - Use ABC for defining interfaces
  - Use composition for flexibility
  - Inject dependencies (don't hardcode)
  - Design for testing
  - Follow "favor composition over inheritance"
""")